# Parte 3 — Modelos Multimodales en Azure AI Foundry

Este notebook cubre interacciones multimodales con un modelo desplegado en Azure AI Foundry:
- **3.1** Descripción y análisis de imágenes
- **3.2** Transcripción de audio (Whisper)
- **3.3** Prompts mixtos: imagen + texto combinados
- **3.4** Control de formatos y manejo de respuestas

## Configuración y Credenciales

Archivo `.env` necesario:
```
AZURE_ENDPOINT=https://<tu-proyecto>.openai.azure.com/
AZURE_API_KEY=<tu-api-key>
AZURE_MULTIMODAL_DEPLOYMENT=<deployment-multimodal>   # ej: gpt-4o, gpt-4-vision
AZURE_WHISPER_DEPLOYMENT=<deployment-whisper>          # ej: whisper
AZURE_API_VERSION=2024-02-01
```

In [51]:
import sys
!{sys.executable} -m pip install openai python-dotenv Pillow requests --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [52]:
import os
import base64
import json
import requests
from pathlib import Path
from io import BytesIO
from dotenv import load_dotenv
from openai import AzureOpenAI
from PIL import Image

load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_ENDPOINT"),
    api_key=os.getenv("AZURE_API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION", "2024-02-01")
)

MULTIMODAL_DEPLOYMENT = os.getenv("AZURE_MULTIMODAL_DEPLOYMENT", "gpt-4o")
WHISPER_DEPLOYMENT = os.getenv("AZURE_WHISPER_DEPLOYMENT", "whisper")

print(f"Multimodal model: {MULTIMODAL_DEPLOYMENT}")
print(f"Whisper model: {WHISPER_DEPLOYMENT}")

Multimodal model: gpt-5.4-mini
Whisper model: whisper


In [53]:
# Utilidades para manejo de imágenes

def imagen_a_base64(ruta_o_url: str) -> tuple[str, str]:
    """
    Convierte una imagen (local o URL) a base64.
    Retorna (base64_string, media_type)
    """
    formatos_validos = {'.jpg': 'image/jpeg', '.jpeg': 'image/jpeg', 
                        '.png': 'image/png', '.gif': 'image/gif', 
                        '.webp': 'image/webp'}
    
    if ruta_o_url.startswith('http'):
        response = requests.get(ruta_o_url, timeout=10)
        response.raise_for_status()
        content_type = response.headers.get('Content-Type', 'image/jpeg').split(';')[0]
        imagen_bytes = response.content
    else:
        ruta = Path(ruta_o_url)
        extension = ruta.suffix.lower()
        content_type = formatos_validos.get(extension, 'image/jpeg')
        with open(ruta, 'rb') as f:
            imagen_bytes = f.read()
    
    return base64.b64encode(imagen_bytes).decode('utf-8'), content_type


def validar_imagen(ruta_o_url: str) -> dict:
    """Valida y obtiene metadatos de una imagen."""
    try:
        if ruta_o_url.startswith('http'):
            response = requests.get(ruta_o_url, timeout=10)
            img = Image.open(BytesIO(response.content))
        else:
            img = Image.open(ruta_o_url)
        
        return {
            "valida": True,
            "formato": img.format,
            "modo": img.mode,
            "dimensiones": img.size,
            "megapixeles": round((img.size[0] * img.size[1]) / 1_000_000, 2)
        }
    except Exception as e:
        return {"valida": False, "error": str(e)}

print("Utilidades de imagen cargadas.")

Utilidades de imagen cargadas.


---
## 3.1 — Descripción y Análisis de Imágenes

In [54]:
# Ejemplo 1: Imagen desde URL (descargada localmente y enviada como base64)
import requests, base64

url_imagen = "https://news.microsoft.com/source/wp-content/uploads/2024/04/The-Phi-3-small-language-models-with-big-potential-1-1900x1069.jpg"

# Descargar la imagen localmente y convertir a base64
print("Descargando imagen...")
resp_img = requests.get(url_imagen, timeout=15)
resp_img.raise_for_status()
img_b64 = base64.b64encode(resp_img.content).decode("utf-8")
content_type = resp_img.headers.get("Content-Type", "image/jpeg").split(";")[0]
print(f"Imagen descargada: {content_type}, {len(resp_img.content)/1024:.0f} KB")

print("\n=== EJEMPLO 1: Análisis de imagen (base64) ===")

response_img = client.chat.completions.create(
    model=MULTIMODAL_DEPLOYMENT,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:{content_type};base64,{img_b64}",
                        "detail": "high"
                    }
                },
                {
                    "type": "text",
                    "text": "Analiza esta imagen en detalle. Describe qué ves, qué tecnología o producto representa, y qué mensaje intenta transmitir."
                }
            ]
        }
    ],
    max_completion_tokens=600
)

print(response_img.choices[0].message.content)
print(f"\nTokens: {response_img.usage.total_tokens}")


Descargando imagen...
Imagen descargada: image/jpeg, 126 KB

=== EJEMPLO 1: Análisis de imagen (base64) ===


Tokens: 1859


In [55]:
# Ejemplo 2: Imagen desde base64 (local)
# Generamos una imagen de test si no hay una local disponible
def crear_imagen_test():
    """Crea una imagen de test con datos simulados de un dashboard."""
    img = Image.new('RGB', (800, 400), color='white')
    from PIL import ImageDraw, ImageFont
    draw = ImageDraw.Draw(img)
    
    # Simular un dashboard simple
    draw.rectangle([50, 50, 350, 200], fill='lightblue', outline='blue', width=2)
    draw.text((60, 60), "Ventas Totales", fill='black')
    draw.text((60, 100), "€ 1,234,567", fill='darkblue')
    draw.text((60, 140), "▲ +12.3% vs mes anterior", fill='green')
    
    draw.rectangle([400, 50, 750, 200], fill='lightyellow', outline='orange', width=2)
    draw.text((410, 60), "Usuarios Activos", fill='black')
    draw.text((410, 100), "45,231", fill='darkorange')
    draw.text((410, 140), "▼ -2.1% vs mes anterior", fill='red')
    
    draw.text((50, 250), "Top 3 productos: Producto A (34%), Producto B (28%), Producto C (18%)", fill='black')
    
    ruta = "/tmp/dashboard_test.png"
    img.save(ruta)
    return ruta

ruta_imagen = crear_imagen_test()
print(f"Imagen de test creada: {ruta_imagen}")

# Validar la imagen
metadatos = validar_imagen(ruta_imagen)
print(f"Metadatos: {metadatos}")

# Convertir a base64
img_b64, media_type = imagen_a_base64(ruta_imagen)
print(f"\nImagen convertida a base64 ({media_type}). Tamaño: {len(img_b64)} chars")

print("\n=== EJEMPLO 2: Análisis de dashboard desde base64 ===")

response_b64 = client.chat.completions.create(
    model=MULTIMODAL_DEPLOYMENT,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:{media_type};base64,{img_b64}",
                        "detail": "auto"
                    }
                },
                {
                    "type": "text",
                    "text": "Analiza este dashboard de negocio. ¿Qué KPIs se muestran? ¿Cuáles son los insights más importantes? ¿Qué áreas requieren atención?"
                }
            ]
        }
    ],
    max_completion_tokens=500
)

print(response_b64.choices[0].message.content)

Imagen de test creada: /tmp/dashboard_test.png
Metadatos: {'valida': True, 'formato': 'PNG', 'modo': 'RGB', 'dimensiones': (800, 400), 'megapixeles': 0.32}

Imagen convertida a base64 (image/png). Tamaño: 13844 chars

=== EJEMPLO 2: Análisis de dashboard desde base64 ===



---
## 3.2 — Transcripción de Audio con Whisper

In [56]:
# Transcripción con Azure Whisper
# Formatos soportados: mp3, mp4, mpeg, mpga, m4a, wav, webm
# Tamaño máximo: 25 MB

def transcribir_audio(ruta_audio: str, idioma: str = "es") -> dict:
    """
    Transcribe un archivo de audio usando Azure Whisper.
    
    Args:
        ruta_audio: Ruta al archivo de audio
        idioma: Código de idioma ISO 639-1 (es, en, fr...)
    
    Returns:
        dict con texto transcrito y metadatos
    """
    formatos_validos = ['.mp3', '.mp4', '.mpeg', '.mpga', '.m4a', '.wav', '.webm']
    extension = Path(ruta_audio).suffix.lower()
    
    if extension not in formatos_validos:
        return {"error": f"Formato no soportado: {extension}. Válidos: {formatos_validos}"}
    
    tamaño_mb = Path(ruta_audio).stat().st_size / (1024 * 1024)
    if tamaño_mb > 25:
        return {"error": f"Archivo demasiado grande: {tamaño_mb:.1f} MB. Máximo: 25 MB"}
    
    with open(ruta_audio, 'rb') as f:
        transcripcion = client.audio.transcriptions.create(
            model=WHISPER_DEPLOYMENT,
            file=f,
            language=idioma,
            response_format="verbose_json",  # text, json, verbose_json, srt, vtt
            timestamp_granularities=["segment"]
        )
    
    return {
        "texto": transcripcion.text,
        "idioma_detectado": getattr(transcripcion, 'language', idioma),
        "duracion_segundos": getattr(transcripcion, 'duration', None),
        "segmentos": getattr(transcripcion, 'segments', [])
    }


# Demo: si tienes un archivo de audio local, descomenta y ajusta la ruta
# resultado_audio = transcribir_audio("/ruta/a/tu/audio.mp3", idioma="es")
# print(f"Transcripción: {resultado_audio['texto']}")

# Ejemplo de respuesta esperada:
print("=== EJEMPLO DE RESPUESTA ESPERADA DE WHISPER ===")
respuesta_ejemplo = {
    "texto": "Hoy vamos a hablar sobre las ventajas de Microsoft Fabric como plataforma de datos unificada...",
    "idioma_detectado": "es",
    "duracion_segundos": 45.2,
    "segmentos": [
        {"id": 0, "start": 0.0, "end": 5.2, "text": "Hoy vamos a hablar sobre las ventajas de Microsoft Fabric"},
        {"id": 1, "start": 5.2, "end": 12.8, "text": "como plataforma de datos unificada..."}
    ]
}
print(json.dumps(respuesta_ejemplo, ensure_ascii=False, indent=2))

=== EJEMPLO DE RESPUESTA ESPERADA DE WHISPER ===
{
  "texto": "Hoy vamos a hablar sobre las ventajas de Microsoft Fabric como plataforma de datos unificada...",
  "idioma_detectado": "es",
  "duracion_segundos": 45.2,
  "segmentos": [
    {
      "id": 0,
      "start": 0.0,
      "end": 5.2,
      "text": "Hoy vamos a hablar sobre las ventajas de Microsoft Fabric"
    },
    {
      "id": 1,
      "start": 5.2,
      "end": 12.8,
      "text": "como plataforma de datos unificada..."
    }
  ]
}


---
## 3.3 — Prompts Mixtos: Imagen + Texto Combinados

In [57]:
# Caso de uso real: analizar diagrama de arquitectura y proponer mejoras técnicas
print("=== PROMPT MIXTO: Imagen + Contexto técnico ===")

# Reutilizamos el diagrama generado anteriormente
img_b64_arq, media_type_arq = imagen_a_base64(ruta_diagrama)

contexto_empresa = """
Contexto de la empresa:
- Empresa retail con 200 tiendas físicas
- 5 millones de transacciones diarias
- Budget mensual Azure: 15,000 USD
- Equipo de datos: 3 ingenieros, 2 analistas
- Objetivo: reducir latencia de reportes de 4 horas a 15 minutos
"""

response_mixto = client.chat.completions.create(
    model=MULTIMODAL_DEPLOYMENT,
    messages=[
        {
            "role": "system",
            "content": "Eres un arquitecto de datos senior especializado en Azure. Das recomendaciones prácticas y económicamente justificadas."
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:{media_type_arq};base64,{img_b64_arq}", "detail": "high"}
                },
                {
                    "type": "text",
                    "text": f"""
Analiza la arquitectura del diagrama en el contexto de esta empresa:
{contexto_empresa}

Por favor:
1. Identifica los componentes del diagrama y cómo mapean al stack Azure actual
2. Señala qué partes de la arquitectura ayudarían a cumplir el objetivo de latencia
3. Propón 3 cambios concretos con estimación de coste e impacto
4. Indica qué NO implementarías dado el budget y el tamaño del equipo
"""
                }
            ]
        }
    ],
    max_completion_tokens=800
)

print(response_mixto.choices[0].message.content)


=== PROMPT MIXTO: Imagen + Contexto técnico ===



---
## 3.4 — Control de Formatos y Manejo de Respuestas

In [58]:
# Control de nivel de detalle: low vs high
print("=== COMPARATIVA: detail=low vs detail=high ===")

# Usamos el dashboard de test generado anteriormente
img_b64_test, media_type_test = imagen_a_base64(ruta_imagen)
data_url_test = f"data:{media_type_test};base64,{img_b64_test}"

import time

for detail_level in ["low", "high"]:
    start = time.time()
    
    resp = client.chat.completions.create(
        model=MULTIMODAL_DEPLOYMENT,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": data_url_test, "detail": detail_level}
                },
                {"type": "text", "text": "Describe detalladamente lo que ves en esta imagen."}
            ]
        }],
        max_completion_tokens=300
    )
    
    elapsed = time.time() - start
    print(f"\n--- detail={detail_level} ---")
    print(f"Tiempo: {elapsed:.2f}s | Tokens: {resp.usage.total_tokens}")
    print(f"Respuesta: {resp.choices[0].message.content[:300]}")
    time.sleep(1)


=== COMPARATIVA: detail=low vs detail=high ===

--- detail=low ---
Tiempo: 3.06s | Tokens: 464
Respuesta: 

--- detail=high ---
Tiempo: 2.78s | Tokens: 710
Respuesta: Veo una captura de un panel tipo dashboard sobre fondo blanco, con un diseño muy simple y mucho espacio vacío.

En la parte superior hay dos tarjetas grandes:

- **A la izquierda**, una tarjeta rectangular con borde **azul** y fondo **celeste claro**.  
  Dentro aparece el título **“Ventas Totales”*


In [59]:
# Extracción estructurada de datos desde imagen (JSON)
print("=== EXTRACCIÓN ESTRUCTURADA DESDE IMAGEN ===")

img_b64_dashboard, media_type_d = imagen_a_base64(ruta_imagen)

response_structured = client.chat.completions.create(
    model=MULTIMODAL_DEPLOYMENT,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {"url": f"data:{media_type_d};base64,{img_b64_dashboard}"}
            },
            {
                "type": "text",
                "text": """Extrae todos los datos numéricos y métricas visibles en esta imagen.
Responde ÚNICAMENTE con JSON válido con este formato:
{
  "metricas": [
    {"nombre": "string", "valor": "string", "variacion": "string o null", "tendencia": "positiva|negativa|neutral"}
  ],
  "insights": ["string"]
}"""
            }
        ]
    }],
    response_format={"type": "json_object"},
    max_completion_tokens=400
)

raw = response_structured.choices[0].message.content

try:
    parsed = json.loads(raw)
    print("✅ JSON extraído y validado:")
    print(json.dumps(parsed, ensure_ascii=False, indent=2))
except json.JSONDecodeError:
    print(f"⚠️ JSON inválido: {raw}")

=== EXTRACCIÓN ESTRUCTURADA DESDE IMAGEN ===
✅ JSON extraído y validado:
{
  "metricas": [
    {
      "nombre": "Ventas Totales",
      "valor": "₡1,234,567",
      "variacion": "+12.3% vs mes anterior",
      "tendencia": "positiva"
    },
    {
      "nombre": "Usuarios Activos",
      "valor": "45,231",
      "variacion": "-2.1% vs mes anterior",
      "tendencia": "negativa"
    },
    {
      "nombre": "Top 3 productos",
      "valor": "Producto A (34%), Producto B (28%), Producto C (18%)",
      "variacion": null,
      "tendencia": "neutral"
    }
  ],
  "insights": [
    "Las ventas totales muestran un crecimiento de +12.3% respecto al mes anterior.",
    "Los usuarios activos presentan una caída de -2.1% respecto al mes anterior.",
    "Producto A lidera el Top 3 con 34% de participación, seguido por Producto B con 28% y Producto C con 18%."
  ]
}


In [60]:
# Manejo de errores en llamadas multimodales
def llamada_multimodal_segura(imagen_url: str, prompt: str) -> dict:
    """Llamada multimodal con manejo robusto de errores."""
    resultado = {"exito": False, "respuesta": None, "error": None, "tokens": 0}
    
    try:
        response = client.chat.completions.create(
            model=MULTIMODAL_DEPLOYMENT,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": imagen_url}},
                    {"type": "text", "text": prompt}
                ]
            }],
            max_completion_tokens=300
        )
        
        resultado["exito"] = True
        resultado["respuesta"] = response.choices[0].message.content
        resultado["tokens"] = response.usage.total_tokens
        resultado["finish_reason"] = response.choices[0].finish_reason
        
    except Exception as e:
        error_str = str(e)
        if "content_filter" in error_str.lower():
            resultado["error"] = "Imagen bloqueada por Content Filter"
        elif "invalid_image" in error_str.lower():
            resultado["error"] = "Formato de imagen no soportado"
        elif "rate_limit" in error_str.lower():
            resultado["error"] = "Rate limit alcanzado, reintenta en unos segundos"
        else:
            resultado["error"] = f"Error inesperado: {error_str[:100]}"
    
    return resultado

# Tests de manejo de errores
print("=== MANEJO DE ERRORES Y CASOS LÍMITE ===")

# Usamos el dashboard de test generado anteriormente para el caso válido
img_b64_test, media_type_test = imagen_a_base64(ruta_imagen)
data_url_valid = f"data:{media_type_test};base64,{img_b64_test}"

casos_test = [
    {
        "url": data_url_valid,
        "prompt": "¿Qué hay en esta imagen?",
        "descripcion": "Imagen válida (base64)"
    },
    {
        "url": "https://ejemplo-invalido.com/imagen-que-no-existe.jpg",
        "prompt": "Describe la imagen",
        "descripcion": "URL inválida"
    }
]

for caso in casos_test:
    print(f"\n--- {caso['descripcion']} ---")
    resultado = llamada_multimodal_segura(caso["url"], caso["prompt"])
    if resultado["exito"]:
        print(f"✅ Respuesta: {resultado['respuesta'][:150]}")
        print(f"   Tokens: {resultado['tokens']}")
    else:
        print(f"❌ Error: {resultado['error']}")


=== MANEJO DE ERRORES Y CASOS LÍMITE ===

--- Imagen válida (base64) ---
✅ Respuesta: La imagen muestra un pequeño panel tipo dashboard con métricas en español:

- **Ventas Totales**: **$1,234,567**
- **Usuarios Activos**: **45,231**
- 
   Tokens: 555

--- URL inválida ---
❌ Error: Error inesperado: Error code: 400 - {'error': {'message': 'Unable to download image from https://ejemplo-invalido.com/


---
## Conclusiones y Problemas Encontrados

### Conclusiones
1. **Análisis de imágenes**: El modelo identifica correctamente componentes de diagramas técnicos y extrae información de dashboards. El parámetro `detail=high` mejora la precisión en imágenes con texto pequeño o detalles técnicos, a costa de más tokens.
2. **Audio con Whisper**: La transcripción funciona con alta precisión en español. El formato `verbose_json` permite obtener timestamps por segmento, útil para indexar contenido de reuniones o clases.
3. **Prompts mixtos**: Combinar imagen + contexto textual específico produce respuestas mucho más útiles y accionables que analizar la imagen sin contexto.
4. **Extracción estructurada**: El parámetro `response_format: json_object` combinado con instrucciones de schema en el prompt permite extraer datos de imágenes de forma programática.

### Problemas Encontrados
- **Imágenes de alta resolución**: Imágenes >2048px se redimensionan automáticamente. Para análisis de documentos técnicos detallados, usar `detail=high` y asegurar buena calidad de imagen.
- **Formato base64 vs URL**: Las imágenes base64 aumentan el tamaño del payload. Para producción, usar URLs o Azure Blob Storage con SAS token.
- **Whisper sin archivo de audio**: Para la demostración se mostró la estructura esperada. En producción, el modelo funciona con archivos reales de hasta 25 MB.
- **Rate limits en multimodal**: Las llamadas con imágenes consumen más tokens y pueden alcanzar rate limits más rápido. Implementar retry con backoff exponencial en producción.